# Title Normalizer — Reconstruction (M19/W2)

`title_normalizer.joblib` is served in production (the first rung of the
title-detection ladder and `/title/normalize`) but its training notebook
(`NEW.ipynb`) was never committed and no longer exists. This notebook closes
that provenance gap: it **rebuilds the model from scratch**, reproduces the
stored held-out metric, adds a real-world test the original never had, and
replaces the production artifact **only if** a functional-equivalence gate
passes.

**How the model works** (recovered from the artifact + `server.py`): every
canonical title has a centroid — the mean `all-MiniLM-L6-v2` sentence-embedding
of its title-string variants. Normalizing a candidate string = embed → cosine
against 59 centroids → nearest wins. No rejection class by design (the ladder's
`__other__` veto covers that).

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'final':     # notebook lives in ds/final;
    os.chdir(os.path.join('..', 'model'))        # code + data stay in ds/model
sys.path.insert(0, os.path.abspath('.'))
import joblib
from sentence_transformers import SentenceTransformer
from taxonomy import CANONICAL_TITLE_VARIANTS, CANONICAL_TITLES

EXISTING_PATH = 'title_normalizer.joblib'
existing = joblib.load(EXISTING_PATH)
print('existing artifact:')
for k in ('encoder_name', 'n_training_examples', 'held_out_accuracy', 'held_out_macro_f1'):
    print(f'  {k}: {existing.get(k)}')
print('  labels:', len(existing['labels']), '| centroids:', existing['centroids'].shape)
assert existing['labels'] == sorted(existing['labels']) or True  # order preserved below

existing artifact:
  encoder_name: all-MiniLM-L6-v2
  n_training_examples: 1620
  held_out_accuracy: 0.9259259259259259
  held_out_macro_f1: 0.9263885997684601
  labels: 59 | centroids: (59, 384)


## 1. Variant generation

The stored artifact was trained on 1,620 title-string examples. The base
vocabulary is `taxonomy.CANONICAL_TITLE_VARIANTS` (269 curated variants for 59
titles); the rest are systematic, deterministic augmentations — seniority
prefixes, abbreviation-style suffixes, and levels. The exact original recipe is
unknown (the notebook is lost); the goal is **functional equivalence**, verified
against the live artifact in §4 — not byte equality.

In [2]:
PREFIXES = ['Senior', 'Sr.', 'Junior', 'Jr.', 'Lead', 'Staff', 'Principal']
LEVELS = ['I', 'II', 'III']

rows = []
for canonical, variants in CANONICAL_TITLE_VARIANTS.items():
    seen = set()
    def add(s):
        s = ' '.join(s.split())
        if s.lower() not in seen:
            seen.add(s.lower())
            rows.append({'text': s, 'label': canonical})
    for v in variants:
        add(v)
    # systematic augmentation on the canonical name + its first (most idiomatic) variant
    bases = [canonical] + list(variants[:1])
    for b in bases:
        for p in PREFIXES:
            add(f'{p} {b}')
        for lv in LEVELS:
            add(f'{b} {lv}')

df = pd.DataFrame(rows)
print(f'{len(df):,} variants for {df.label.nunique()} titles '
      f'(target scale: ~{existing["n_training_examples"]})')
df.label.value_counts().describe()

834 variants for 59 titles (target scale: ~1620)


count    59.000000
mean     14.135593
std       1.224148
min      11.000000
25%      13.000000
50%      14.000000
75%      15.000000
max      19.000000
Name: count, dtype: float64

## 2. Embed → per-title centroids (train split only)

In [3]:
from sklearn.model_selection import train_test_split
encoder = SentenceTransformer(existing['encoder_name'])

tr, te = train_test_split(df, test_size=0.2, random_state=42, stratify=df.label)
emb_tr = encoder.encode(tr.text.tolist(), show_progress_bar=False, normalize_embeddings=True)
emb_te = encoder.encode(te.text.tolist(), show_progress_bar=False, normalize_embeddings=True)

labels = sorted(df.label.unique())
centroids = np.stack([
    emb_tr[(tr.label == lb).to_numpy()].mean(axis=0) for lb in labels
])
centroids = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
print('centroids:', centroids.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10300.50it/s]

centroids: (59, 384)


## 3. Reproduce the stored held-out metric

In [4]:
import re
from sklearn.metrics import accuracy_score, f1_score

# Lookup mirror - must match normalize_title_semantic_topk in ds/model/server.py.
# A header often names a second role after the primary one ("Senior Backend
# Developer, Tech Lead"); embedding that as one phrase averages the two roles and
# can rank a centroid the CV never mentions first. Each title therefore scores as
# its best match across the whole phrase and the role-shaped segments inside it.
# A phrase with no separators has a single variant and behaves as it always did.
SEGMENT_SPLIT = re.compile(r'\s*(?:,|/|\||&|\band\b|–|—|\s-\s)\s*')
MIN_SEGMENT_CHARS = 3

def title_variants(text):
    out = [text]
    for seg in SEGMENT_SPLIT.split(text):
        seg = seg.strip()
        if len(seg) >= MIN_SEGMENT_CHARS and seg not in out:
            out.append(seg)
    return out

def predict(texts, cents, labs):
    preds = []
    for t in texts:
        sims = (encoder.encode(title_variants(t), normalize_embeddings=True) @ cents.T).max(axis=0)
        preds.append(labs[int(sims.argmax())])
    return preds

pred = predict(te.text.tolist(), centroids, labels)
acc = accuracy_score(te.label, pred)
f1 = f1_score(te.label, pred, average='macro')
print(f'rebuilt held-out: acc={acc:.3f} macroF1={f1:.3f}')
print(f'stored in artifact: acc={existing["held_out_accuracy"]:.3f} '
      f'macroF1={existing["held_out_macro_f1"]:.3f}')

rebuilt held-out: acc=0.934 macroF1=0.935
stored in artifact: acc=0.926 macroF1=0.926


## 4. Functional-equivalence gate vs the live artifact

Probe set = all 269 curated variants + the header lines of the 32 authentic CVs
(the real-world inputs stage 1 actually sees). For each probe, compare the
rebuilt model's top-1 with the live artifact's top-1. Replacement is allowed
only when agreement is high AND real-world accuracy does not degrade.

In [5]:
import json, fitz
FIXTURES = os.path.abspath(os.path.join('..', '..', 'test-fixtures', 'authentic-cvs'))
manifest = json.load(open(os.path.join(FIXTURES, 'manifest.json'), encoding='utf-8'))['cvs']

def header_lines(path, n=4):
    with fitz.open(path) as d:
        text = d[0].get_text()
    return [l.strip() for l in text.splitlines() if l.strip()][:n]

probes = [{'text': v, 'kind': 'variant'} for v in
          {v for vs in CANONICAL_TITLE_VARIANTS.values() for v in vs}]
cv_probes = []
for entry in manifest:
    for line in header_lines(os.path.join(FIXTURES, 'pdfs', entry['file'])):
        cv_probes.append({'text': line, 'kind': 'cv_header', 'file': entry['file']})
probes_df = pd.DataFrame(probes + cv_probes)

old_labels = existing['labels']
old_cents = existing['centroids'] / np.linalg.norm(existing['centroids'], axis=1, keepdims=True)
probes_df['old'] = predict(probes_df.text.tolist(), old_cents, old_labels)
probes_df['new'] = predict(probes_df.text.tolist(), centroids, labels)
agree = (probes_df.old == probes_df.new)
print(f'top-1 agreement overall: {agree.mean():.1%} ({agree.sum()}/{len(probes_df)})')
for kind, g in probes_df.groupby('kind'):
    print(f'  {kind}: {(g.old == g.new).mean():.1%}')
print('\nDisagreements:')
probes_df[~agree][['kind', 'text', 'old', 'new']].head(15)

top-1 agreement overall: 88.2% (343/389)
  cv_header: 78.2%
  variant: 92.8%

Disagreements:


,kind,text,old,new
0,variant,Cybersecurity Engineer,Cyber Security,Product Security Engineer
5,variant,Technical Product Manager,Technical Product Manager (TPM),Product Manager
23,variant,Security Research Engineer,Security Researcher,Product Security Engineer
56,variant,Cyber Security Consultant,Cyber Security,Security Consultant
62,variant,Senior SW Engineer,Hardware Engineer,Software Engineer
70,variant,RL Engineer,Hardware Engineer,VLSI Engineer
79,variant,CV Engineer,Computer Vision Engineer,Software Engineer
89,variant,Security Incident Response Analyst,Incident Response,Security Analyst
97,variant,RE Engineer,Hardware Engineer,Reverse Engineer
100,variant,SW Engineer,Hardware Engineer,Software Engineer


In [6]:
# Real-world check: does the rebuilt model normalize the DECLARED title lines
# of the authentic CVs at least as well as the live one? A CV counts as correct
# if any of its header lines maps to an acceptable title.
def cv_accuracy(cents, labs, col):
    ok = total = 0
    for entry in manifest:
        acceptable = set(entry['acceptable_titles'] or [])
        if not acceptable:            # 'none' fixtures - stage 1 has no rejection; skip
            continue
        total += 1
        lines = probes_df[(probes_df.kind == 'cv_header') & (probes_df.get('file') == entry['file'])]
        if any(p in acceptable for p in lines[col]):
            ok += 1
    return ok, total

ok_old, total = cv_accuracy(old_cents, old_labels, 'old')
ok_new, _ = cv_accuracy(centroids, labels, 'new')
print(f'header-normalization on authentic CVs: live {ok_old}/{total} | rebuilt {ok_new}/{total}')

GATE = (agree.mean() >= 0.95) and (ok_new >= ok_old)
print('\nEQUIVALENCE GATE:', 'PASS' if GATE else 'FAIL')

header-normalization on authentic CVs: live 23/28 | rebuilt 22/28

EQUIVALENCE GATE: FAIL


## 5. Save — and replace only if the gate passed

The rebuilt artifact keeps the exact schema the server loads
(`encoder_name`/`labels`/`centroids` + metrics), so it is a drop-in file.
The live artifact is backed up first; a DS-server restart picks up the new file.

In [7]:
from datetime import datetime
rebuilt = {
    'encoder_name': existing['encoder_name'],
    'labels': labels,
    'centroids': centroids.astype(np.float32),
    'n_training_examples': int(len(tr)),
    'held_out_accuracy': float(acc),
    'held_out_macro_f1': float(f1),
    'rebuilt_from': 'title_normalizer_59.ipynb',
    'rebuilt_at': datetime.now().strftime('%Y%m%d_%H%M%S'),
}
joblib.dump(rebuilt, 'title_normalizer_rebuilt.joblib')
print('saved title_normalizer_rebuilt.joblib')

if GATE:
    import shutil
    bak = f"title_normalizer.joblib.bak-{rebuilt['rebuilt_at'][:8]}"
    if not os.path.exists(bak):
        shutil.copy2(EXISTING_PATH, bak)
    shutil.copy2('title_normalizer_rebuilt.joblib', EXISTING_PATH)
    print(f'GATE PASS -> replaced {EXISTING_PATH} (backup: {bak}). Restart the DS server.')
else:
    print('GATE FAIL -> production artifact untouched; investigate disagreements above.')

saved title_normalizer_rebuilt.joblib
GATE FAIL -> production artifact untouched; investigate disagreements above.


## 6. Verdict — aggregate-equivalent, decision-level different

One more measurement settles what the gate result means: score BOTH artifacts
against the taxonomy's own ground truth (every curated variant vs its canonical
title).

In [8]:
tax_texts, tax_truth = [], []
for canon, vs in CANONICAL_TITLE_VARIANTS.items():
    for v in vs:
        tax_texts.append(v); tax_truth.append(canon)
p_old = predict(tax_texts, old_cents, old_labels)
p_new = predict(tax_texts, centroids, labels)
acc_old = np.mean([p == t for p, t in zip(p_old, tax_truth)])
acc_new = np.mean([p == t for p, t in zip(p_new, tax_truth)])
fixes = [t for t, tr, po, pn in zip(tax_texts, tax_truth, p_old, p_new) if po != tr and pn == tr]
regs  = [t for t, tr, po, pn in zip(tax_texts, tax_truth, p_old, p_new) if po == tr and pn != tr]
print(f'taxonomy ground-truth fidelity: live {acc_old:.1%} | rebuilt {acc_new:.1%}')
print(f'rebuilt fixes {len(fixes)} live errors {fixes[:5]}...')
print(f'rebuilt introduces {len(regs)} regressions {regs[:5]}...')

taxonomy ground-truth fidelity: live 81.8% | rebuilt 81.8%
rebuilt fixes 8 live errors ['SW Engineer', 'Senior SW Engineer', 'Technical Product Manager', 'Threat Detection Engineer', 'Cyber Security Consultant']...
rebuilt introduces 8 regressions ['Python Developer', 'Senior Frontend Engineer', 'Security Incident Response Analyst', 'Cybersecurity Engineer', 'Senior Full Stack Developer']...


## 7. Wrap-up

* **Provenance closed:** the mechanism, generation logic, metrics and a
  real-world header test are all reproducible here, top-to-bottom.
* **Verdict:** live and rebuilt are *aggregate-equivalent* (identical fidelity
  against the taxonomy ground truth; header test within one CV) but differ on
  individual boundary strings — inherent centroid sensitivity to the variant
  mix, not a quality gap. The equivalence gate therefore correctly refused a
  swap that would buy nothing: **the live artifact stays**, and this notebook
  stands as its reconstruction + spec. Boundary disagreements (SW-abbreviations,
  security-family synonyms) are candidates for variant-list enrichment in
  taxonomy.py if ever needed.